In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [4]:
movies = pd.read_csv("movies.csv", index_col='movieId')
ratings = pd.read_csv("ratings.csv", index_col='userId')

In [5]:
movies.head()

,title,genres
movieId,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy


In [6]:
ratings.tail()

,movieId,rating,timestamp
userId,,,
610,166534,4.0,1493848402
610,168248,5.0,1493850091
610,168250,5.0,1494273047
610,168252,5.0,1493846352
610,170875,3.0,1493846415


In [7]:
print(movies.shape)
print(ratings.shape)

(9742, 2)
(100836, 3)


In [8]:
df=pd.merge(ratings, movies, on = 'movieId')
df.head()

,movieId,rating,timestamp,title,genres
0,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   movieId    100836 non-null  int64  
 1   rating     100836 non-null  float64
 2   timestamp  100836 non-null  int64  
 3   title      100836 non-null  object 
 4   genres     100836 non-null  object 
dtypes: float64(1), int64(2), object(2)
memory usage: 3.8+ MB


In [10]:
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df['title'] = df['title'].astype(str)
df['genres'] = df['genres'].astype(str)

In [11]:
Popularity = df.groupby('title')['rating'].agg(['count','mean'])
Popularity.columns = ['avg_rating', 'rating_count']
Popularity.head()

,avg_rating,rating_count
title,,
'71 (2014),1,4.0
'Hellboy': The Seeds of Creation (2004),1,4.0
'Round Midnight (1986),2,3.5
'Salem's Lot (2004),1,5.0
'Til There Was You (1997),2,4.0


In [12]:
top_popular = Popularity.sort_values(by='rating_count', ascending=False).head(5)
top_popular

,avg_rating,rating_count
title,,
Karlson Returns (1970),1,5.0
Zeitgeist: Moving Forward (2011),1,5.0
"Dream of Light (a.k.a. Quince Tree Sun, The) (Sol del membrillo, El) (1992)",1,5.0
Dragons: Gift of the Night Fury (2011),1,5.0
12 Angry Men (1997),1,5.0


In [13]:
similarity_matrix = cosine_similarity(Popularity)

user_similarity = pd.DataFrame(similarity_matrix,
                               index=Popularity.index,   # movie IDs
                               columns=Popularity.index) # movie IDs

print("Item (Movie) Similarity Matrix:\n", user_similarity)



Item (Movie) Similarity Matrix:
 title                                      '71 (2014)  \
title                                                   
'71 (2014)                                   1.000000   
'Hellboy': The Seeds of Creation (2004)      1.000000   
'Round Midnight (1986)                       0.962651   
'Salem's Lot (2004)                          0.998868   
'Til There Was You (1997)                    0.976187   
...                                               ...   
eXistenZ (1999)                              0.406688   
xXx (2002)                                   0.352200   
xXx: State of the Union (2005)               0.585491   
¡Three Amigos! (1986)                        0.356914   
À nous la liberté (Freedom for Us) (1931)    0.857493   

title                                      'Hellboy': The Seeds of Creation (2004)  \
title                                                                                
'71 (2014)                                           

In [14]:
movies['genres_processed']=movies['genres'].fillna('').str.replace('|', ' ')

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [16]:

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_processed'])


In [17]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [18]:
def get_recommendations(title, movies, cosine_sim):
        
    indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:6]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return movies[['title', 'genres']].iloc[movie_indices]

In [19]:
recommendations = get_recommendations('Toy Story (1995)', movies, cosine_sim)
print("Top 5 movies similar to Toy Story (1995):")
print(recommendations)

Top 5 movies similar to Toy Story (1995):
                                             title                      genres
movieId                                                                       
60              Indian in the Cupboard, The (1995)  Adventure|Children|Fantasy
126              NeverEnding Story III, The (1994)  Adventure|Children|Fantasy
1009               Escape to Witch Mountain (1975)  Adventure|Children|Fantasy
2043     Darby O'Gill and the Little People (1959)  Adventure|Children|Fantasy
2093                           Return to Oz (1985)  Adventure|Children|Fantasy


In [20]:
import streamlit as st
import pickle

In [23]:
pickle.dump(df.to_dict(), open('movies_dict.pkl','wb'))
pickle.dump(tfidf, open('model.pkl','wb'))
pickle.dump(tfidf_matrix, open('genre_matrix.pkl','wb'))